# AeroFleet — 1,000-case mass forensics run on Kaggle (free GPU)

Runs the real (non-mock) LLM evaluation harness against a free Kaggle P100/T4, chunked across
sessions using the harness's own per-case checkpoint (`results.jsonl`). Safe to stop and resume —
each session picks up exactly where the last one left off.

## Required one-time manual setup (Kaggle UI, not this notebook)

1. **Settings → Accelerator → GPU P100** (or T4 x2). Settings → Internet → **On**.
2. **Add-ons → Secrets → add a secret named `GH_PAT`**: a GitHub fine-grained Personal Access
   Token scoped to **read-only, this repo only** (`AdityaPathare46/aerofleet`, which is private).
   Never put the token in a cell — it's read from the Secrets store at runtime only.
3. Session cap is ~9h and Kaggle gives ~30 GPU-hours/week. When a session ends (timeout, quota,
   or you stopping it), **Save Version (commit)** before closing — that's what persists
   `/kaggle/working` as this notebook's Output for the next session to resume from.

## Known, deliberate deviation from the real (college-PC) run — put this in the paper

`llama4:scout` is 67GB (109B-param MoE) and doesn't fit any free-tier GPU (16GB VRAM). It powers
4 agents: DISPATCHER, AIRSPACE_SAFETY, AI_VALIDATOR, CONTINGENCY. For this cloud run only, those
4 agents use **`mistral-nemo:12b`** (Mistral AI, 12B, non-Chinese-origin — same constraint the
project's hardware roster already applies) instead, via the existing `AGENT_MODEL_<AGENT_ID>`
env-var override (`aerofleet/agents/factory.py`) — no code changes. This notebook writes a
`cloud_run_metadata.json` recording the substitution, date, and GPU used, so the deviation is
traceable rather than silently baked into the numbers. Report this run's results as a **secondary/
robustness** result, not a drop-in replacement for the primary college-PC run with the real roster.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 1. Clone the private repo

Uses the `GH_PAT` secret — the token is interpolated into the clone URL and never printed or
stored in this notebook.


In [ ]:
from kaggle_secrets import UserSecretsClient
_token = UserSecretsClient().get_secret("GH_PAT")

REPO_DIR = "/kaggle/working/aerofleet"
import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://{_token}@github.com/AdityaPathare46/aerofleet.git {REPO_DIR}
else:
    print("Repo already present, skipping clone.")

del _token  # don't leave it bound in the kernel namespace longer than needed
%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 2. Install Ollama and start the server in this session


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, requests

log = open("/kaggle/working/ollama_serve.log", "a")
proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not come up — check /kaggle/working/ollama_serve.log")


## 3. Pull the models

Three real roster models plus the `mistral-nemo:12b` substitute (~41GB total). If a pull fails or
this cell is interrupted, just re-run it — Ollama resumes partial downloads.


In [ ]:
for model in ["gemma4:12b", "phi4-reasoning:plus", "mistral-small3.2", "mistral-nemo:12b"]:
    print(f"--- pulling {model} ---")
    !ollama pull {model}

!ollama list


## 4. Resume from a previous session's checkpoint (if any)

If you saved a prior session's output and re-attached it as an input dataset (Add Data → Your
Work → this notebook's earlier Output), point `PREVIOUS_RESULTS_DIR` at it — its `results.jsonl`
gets copied into this session's study dir so the harness resumes instead of starting over.
Leave as `None` for the very first session.


In [ ]:
import shutil
from pathlib import Path

STUDY_DIR = Path("/kaggle/working/mass_forensics_kaggle")
STUDY_DIR.mkdir(parents=True, exist_ok=True)

PREVIOUS_RESULTS_DIR = None  # e.g. "/kaggle/input/aerofleet-kaggle-mass-forensics-run/mass_forensics_kaggle"

if PREVIOUS_RESULTS_DIR is not None:
    prev = Path(PREVIOUS_RESULTS_DIR)
    for fname in ["results.jsonl", "run_config.json"]:
        src = prev / fname
        if src.exists():
            shutil.copy(src, STUDY_DIR / fname)
            print(f"Restored {fname} from previous session")
else:
    print("No previous session pointed to — starting a fresh study.")


## 5. Configure the model substitution and record it

`AGENT_MODEL_<AGENT_ID>` overrides `AgentFactory.DEFAULT_MODEL_MAP` per-agent, no code edits
needed (`aerofleet/agents/factory.py:91-94`).


In [ ]:
import os, json, subprocess, datetime

os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ.pop("USE_MOCK_AGENTS", None)  # make sure we are NOT in mock mode

for agent_id in ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"]:
    os.environ[f"AGENT_MODEL_{agent_id}"] = "mistral-nemo:12b"

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

metadata = {
    "platform": "kaggle",
    "gpu": gpu_name,
    "run_started_utc": datetime.datetime.utcnow().isoformat(),
    "substitution": {
        "replaced_model": "llama4:scout",
        "substitute_model": "mistral-nemo:12b",
        "reason": "llama4:scout is 67GB (109B-param MoE); does not fit a free-tier 16GB GPU",
        "affected_agents": ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"],
    },
    "unaffected_agents_real_roster": {
        "ROUTE": "mistral-small3.2", "COMMS": "mistral-small3.2", "COMPLIANCE": "mistral-small3.2",
        "BATTERY": "phi4-reasoning:plus", "COST": "phi4-reasoning:plus", "PAYLOAD": "phi4-reasoning:plus",
        "WEATHER": "gemma4:12b", "OPS": "gemma4:12b",
    },
}
with open(STUDY_DIR / "cloud_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 6. Run the harness for this session

No `--max-batches` cap — it just runs until the 1,000-case target is hit, Kaggle's session limit
cuts it off, or you interrupt it. `results.jsonl` is appended to per-case, so an interruption loses
at most one in-flight case, never a whole batch. `--batch-size 25` (smaller than the 250 default)
gives more frequent progress checkpoints for a session this short.


In [ ]:
!python -m scenario_engine.mass_forensics_evaluation \
    --study-dir {STUDY_DIR} \
    --target 1000 \
    --batch-size 25


## 7. Check progress at any point


In [ ]:
import json
results_file = STUDY_DIR / "results.jsonl"
if results_file.exists():
    with open(results_file) as f:
        n = sum(1 for _ in f)
    print(f"{n} case-attempts recorded so far (target: 1000 cases; retries count as separate attempts).")
else:
    print("No results yet.")


## 8. End of session — persist for next time

Click **Save Version → Save & Run All (Commit)** in the Kaggle UI. That snapshots everything under
`/kaggle/working` (including `results.jsonl` and `cloud_run_metadata.json`) as this notebook's
Output. Next session: **Add Data → Your Work → (this notebook, latest version)**, then set
`PREVIOUS_RESULTS_DIR` in the cell above to the mounted path Kaggle shows for it, and re-run from
the top. Repeat weekly until `results.jsonl` covers all 1,000 cases.

Once it's done, pull the finished `results.jsonl` + generated report down and drop the real numbers
into `research/baseline_comparison.md`-style summary — the same pattern already used for the mock
baseline — rather than just linking to it, per `research/reproducibility.md`'s existing convention.
